<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/MCP_OpenWeather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement
**Problem Statement**

Build a **pure MCP-based Weather Assistant** that allows an AI model to access real-time weather information through an external tool instead of relying on static or pre-trained knowledge.

The system should expose an MCP tool called `get_current_weather`, which connects to the OpenWeather API and retrieves live weather details for a given city. The tool should return information such as temperature, weather condition, humidity, wind speed, and feels-like temperature.

The goal is to separate the AI reasoning layer from the real-time data access layer. The MCP server will handle communication with the weather API, while the AI assistant will discover and call the weather tool whenever the user asks for current weather information.

This makes the assistant more accurate, modular, reusable, and production-ready for real-time weather queries.


In [ ]:
!pip install -q fastmcp anthropic requests

import json
import os
import requests
from anthropic import Anthropic
from fastmcp import FastMCP
from fastmcp.client import Client


anthropic_client = Anthropic(
    api_key= input("Anthropic_API_KEY: ")

)
OPENWEATHER_API_KEY = input("OPENWEATHER_API_KEY: ")


# ---------------------------------------
# 2. Create MCP Server
# ---------------------------------------

mcp_server = FastMCP("OpenWeather MCP Server")


# ---------------------------------------
# 3. Create MCP Tool
# ---------------------------------------

@mcp_server.tool()
def get_current_weather(city: str, units: str = "metric") -> str:
    """
    Get current weather for a city using OpenWeather API.
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": OPENWEATHER_API_KEY,
        "units": units
    }

    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200:
        return json.dumps({
            "error": data.get("message", "Unable to fetch weather")
        }, indent=2)

    weather_result = {
        "city": data["name"],
        "country": data["sys"]["country"],
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "condition": data["weather"][0]["description"],
        "wind_speed": data["wind"]["speed"],
        "units": units
    }

    return json.dumps(weather_result, indent=2)



    # ---------------------------------------
# 4. Run Claude Agent with MCP Tool
# ---------------------------------------

async def run_agent(user_question: str):

    async with Client(mcp_server) as mcp_client:

        # Get tools from MCP server
        mcp_tools = await mcp_client.list_tools()

        # Convert MCP tools into Claude tool format
        claude_tools = [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.inputSchema
            }
            for tool in mcp_tools
        ]

        messages = [
            {"role": "user", "content": user_question}
        ]

        while True:

            response = anthropic_client.messages.create(
                model="claude-sonnet-4-5-20250929",
                max_tokens=700,
                tools=claude_tools,
                messages=messages
            )

            # If Claude gives final answer, print and stop
            if response.stop_reason != "tool_use":
                print(response.content[0].text)
                return
            '''
            # Get the tool Claude wants to use
            tool_request = next(
                block for block in response.content
                if block.type == "tool_use"
            )
            '''
            tool_request = response.content[0]


            # Run the MCP tool
            tool_result = await mcp_client.call_tool(
                tool_request.name,
                arguments=tool_request.input
            )

            # Add Claude's tool request to conversation
            messages.append({
                "role": "assistant",
                "content": response.content
            })

            # Add tool output back to Claude
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_request.id,
                        "content": tool_result.content[0].text
                    }
                ]
            })

In [7]:
await run_agent(" current weather in dwarika in celcius")

The current weather in Dwarka, India:

- **Temperature:** 28.9°C
- **Feels like:** 34.5°C
- **Condition:** Clear sky
- **Humidity:** 81%
- **Wind speed:** 7.5 m/s

It's quite warm with high humidity, making it feel hotter than the actual temperature!
